# Weather Data Scraper & Analyzer

**Goal:** Scrape live weather data for a city from the free
[wttr.in](https://wttr.in) service (no API key needed), log it over time, and
analyze temperature, humidity, wind, and rain-chance trends.

This notebook covers:

1. Scraping live weather data (current conditions + 3-day forecast)
2. Logging snapshots to a local CSV to build a real history over time
3. Turning the forecast into a clean, analyzable DataFrame
4. Visualizing temperature, humidity, wind, and rain-chance trends
5. Key insights & possible extensions

**City used:** Chennai (change the `CITY` variable to track any other city)


## 1. Setup

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os
import json

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

CITY = "Chennai"
LOG_FILE = "weather_log.csv"


## 2. The Scraper

`wttr.in` returns rich JSON (current conditions + a 3-day hourly forecast) when
you request `https://wttr.in/{city}?format=j1` — no sign-up, no API key.

The function below fetches live data. If there's no internet connection (or the
service is briefly unreachable), it automatically falls back to a clearly-labeled
demo dataset with the *exact same structure*, so the rest of the notebook still
runs end-to-end. Delete the fallback once you've confirmed your machine has
internet access.

In [ ]:
def generate_demo_data(city):
    """Fallback data with the same schema as a real wttr.in response.
    Used only when live data can't be reached (e.g. no internet)."""
    rng = np.random.default_rng(42)
    now = datetime.now()

    current_condition = [{
        "temp_C": "31",
        "FeelsLikeC": "38",
        "humidity": "68",
        "windspeedKmph": "14",
        "pressure": "1008",
        "visibility": "6",
        "uvIndex": "7",
        "weatherDesc": [{"value": "Partly cloudy"}],
        "localObsDateTime": now.strftime("%Y-%m-%d %I:%M %p"),
    }]

    weather = []
    base_temp = 30
    for d in range(3):
        hourly = []
        for h in range(8):  # wttr.in reports every 3 hours -> 8 slots/day
            hour_temp = base_temp + 4 * np.sin(np.pi * h / 8) + rng.normal(0, 1)
            hourly.append({
                "time": str(h * 300).zfill(4)[:2] + "00" if h else "0",
                "tempC": str(round(hour_temp, 1)),
                "humidity": str(int(np.clip(rng.normal(70, 8), 40, 95))),
                "windspeedKmph": str(int(np.clip(rng.normal(13, 5), 2, 35))),
                "chanceofrain": str(int(np.clip(rng.normal(20, 20), 0, 100))),
                "weatherDesc": [{"value": rng.choice(
                    ["Sunny", "Partly cloudy", "Cloudy", "Patchy rain possible"]
                )}],
            })
        weather.append({
            "date": (now + pd.Timedelta(days=d)).strftime("%Y-%m-%d"),
            "maxtempC": str(round(base_temp + 5, 1)),
            "mintempC": str(round(base_temp - 4, 1)),
            "avgtempC": str(round(base_temp, 1)),
            "hourly": hourly,
            "astronomy": [{"sunrise": "05:58 AM", "sunset": "06:42 PM"}],
        })
        base_temp += rng.normal(0, 1.5)

    return {
        "current_condition": current_condition,
        "weather": weather,
        "nearest_area": [{
            "areaName": [{"value": city}],
            "country": [{"value": "India"}],
        }],
    }


def fetch_weather_data(city):
    url = f"https://wttr.in/{city}?format=j1"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return response.json(), "live"
    except Exception as e:
        print(f"Could not reach wttr.in ({e}).")
        print("Falling back to demo data so the notebook still runs end-to-end.")
        return generate_demo_data(city), "demo"


raw_data, source = fetch_weather_data(CITY)
print(f"Data source: {source.upper()}")
print(json.dumps(raw_data["current_condition"][0], indent=2))


## 3. Logging Snapshots Over Time

Each time you run this notebook (e.g. once a day, or on a schedule via cron / Task Scheduler), it appends the current snapshot to `weather_log.csv`. Run it regularly for a few days and you'll have a genuine historical dataset to analyze, instead of just a single point in time.

In [ ]:
current = raw_data["current_condition"][0]
area = raw_data["nearest_area"][0]["areaName"][0]["value"]

snapshot = {
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "city": area,
    "temp_C": float(current["temp_C"]),
    "feels_like_C": float(current["FeelsLikeC"]),
    "humidity": int(current["humidity"]),
    "windspeed_kmph": float(current["windspeedKmph"]),
    "pressure": float(current["pressure"]),
    "uv_index": float(current["uvIndex"]),
    "description": current["weatherDesc"][0]["value"],
    "source": source,
}

log_df = pd.DataFrame([snapshot])

if os.path.exists(LOG_FILE):
    log_df.to_csv(LOG_FILE, mode="a", header=False, index=False)
else:
    log_df.to_csv(LOG_FILE, mode="w", header=True, index=False)

history = pd.read_csv(LOG_FILE)
print(f"Log now has {len(history)} snapshot(s).")
history.tail()


## 4. Building a Forecast DataFrame

`wttr.in` also returns a 3-day hourly forecast in a single response — enough to demonstrate trend analysis even on the very first run.

In [ ]:
rows = []
for day in raw_data["weather"]:
    date = day["date"]
    for hour in day["hourly"]:
        hour_of_day = int(hour["time"]) // 100
        rows.append({
            "date": date,
            "hour": hour_of_day,
            "datetime_label": f"{date} {hour_of_day:02d}:00",
            "temp_C": float(hour["tempC"]),
            "humidity": int(hour["humidity"]),
            "windspeed_kmph": float(hour["windspeedKmph"]),
            "chance_of_rain": int(hour["chanceofrain"]),
            "description": hour["weatherDesc"][0]["value"],
        })

forecast_df = pd.DataFrame(rows)
forecast_df.head(10)


## 5. Visualizing Trends

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(forecast_df["datetime_label"], forecast_df["temp_C"], marker="o", color="#e67e22")
plt.title(f"Temperature Trend — {area} (next 3 days)")
plt.xlabel("Date & Hour")
plt.ylabel("Temperature (°C)")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(forecast_df["datetime_label"], forecast_df["humidity"], marker="o", color="#2980b9")
axes[0].set_title("Humidity Trend")
axes[0].set_ylabel("Humidity (%)")
axes[0].tick_params(axis="x", rotation=75)

axes[1].plot(forecast_df["datetime_label"], forecast_df["windspeed_kmph"], marker="o", color="#27ae60")
axes[1].set_title("Wind Speed Trend")
axes[1].set_ylabel("Wind Speed (kmph)")
axes[1].tick_params(axis="x", rotation=75)

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
daily_rain = forecast_df.groupby("date")["chance_of_rain"].mean().round(1)
daily_rain.plot(kind="bar", color="#8e44ad")
plt.title("Average Chance of Rain by Day")
plt.ylabel("Chance of Rain (%)")
plt.xticks(rotation=0)
plt.show()


In [ ]:
plt.figure(figsize=(7, 6))
sns.scatterplot(data=forecast_df, x="temp_C", y="humidity", hue="date", s=80)
plt.title("Temperature vs Humidity")
plt.xlabel("Temperature (°C)")
plt.ylabel("Humidity (%)")
plt.show()

correlation = forecast_df["temp_C"].corr(forecast_df["humidity"])
print(f"Correlation between temperature and humidity: {correlation:.2f}")


## 6. Summary Statistics

In [ ]:
summary = forecast_df.groupby("date").agg(
    avg_temp=("temp_C", "mean"),
    max_temp=("temp_C", "max"),
    min_temp=("temp_C", "min"),
    avg_humidity=("humidity", "mean"),
    avg_windspeed=("windspeed_kmph", "mean"),
    avg_rain_chance=("chance_of_rain", "mean"),
).round(1)

summary


## 7. Key Insights

- **Temperature follows a clear daily cycle** — rising through the day and
  dropping overnight, visible in the 3-day trend line.
- **Temperature and humidity are typically inversely related** — the
  correlation printed above is usually negative, since hotter hours tend to be
  drier.
- **Rain chance varies noticeably by day**, which is exactly the kind of signal
  you'd want before planning outdoor activities.
- **The historical log (`weather_log.csv`) grows every run** — after a week of
  daily runs you'll have real day-over-day data instead of just a forecast
  snapshot, enabling actual historical trend analysis (not just forecast trends).

## 8. Possible Next Steps

- Automate this notebook/script to run daily via cron (Linux/Mac) or Task
  Scheduler (Windows) to build a genuine multi-week historical dataset.
- Track multiple cities in the same log and compare climates.
- Add anomaly detection (e.g. flag days where temperature deviates sharply
  from the recent rolling average).
- Feed the historical log into a simple forecasting model (e.g. moving average
  or ARIMA) and compare predictions against wttr.in's own forecast.
